# 图片批量推理测试

- 使用 `examples/images.md` 作为模型 instruction。
- 从 `/home/dream/ProjectDATA/Testset/validation_dataset_human_recognition` 逐张读取图片。
- 将 JSON 结果按列展开后保存到 `examples/output/image_instruction_results.csv`。

In [ ]:
from __future__ import annotations

import json
import time
from base64 import b64encode
from pathlib import Path

import pandas as pd
from openai import OpenAI

PROJECT_ROOT = Path("/home/dream/Study/26p1/DSproject/ProjectNew")
PROMPT_PATH = PROJECT_ROOT / "examples" / "images.md"
IMAGE_DIR = Path("/home/dream/ProjectDATA/Testset/images")
OUTPUT_CSV = PROJECT_ROOT / "examples" / "output" / "image_instruction_results3-5.csv"

BASE_URL = "http://localhost:8000/v1"
API_KEY = "EMPTY"
MODEL_NAME = "Qwen/Qwen3.5-4B"
MAX_IMAGES = 300
TEMPERATURE = 0.7
MAX_TOKENS = 4096
REQUEST_INTERVAL_SECONDS = 0.0
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp"}
KEEP_IMAGE_PATH_COLUMN = False


def load_instruction(path: Path) -> str:
    return path.read_text(encoding="utf-8").strip()


def image_file_to_data_url(image_path: str) -> str:
    path = Path(image_path)
    mime_type_map = {
        ".jpg": "image/jpeg"
    }
    mime_type = mime_type_map.get(path.suffix.lower(), "application/octet-stream")
    encoded = b64encode(path.read_bytes()).decode("utf-8")
    return f"data:{mime_type};base64,{encoded}"


def message_content_to_text(content) -> str:
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts: list[str] = []
        for item in content:
            if isinstance(item, dict):
                text = item.get("text")
                if text:
                    parts.append(str(text))
            elif item is not None:
                parts.append(str(item))
        return "\n".join(parts)
    return str(content)


def extract_json_text(raw_text: str) -> str:
    cleaned = raw_text.strip()
    if not cleaned:
        raise ValueError("模型返回为空")

    if cleaned.startswith("```"):
        lines = cleaned.splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].startswith("```"):
            lines = lines[:-1]
        cleaned = "\n".join(lines).strip()
        if cleaned.lower().startswith("json"):
            cleaned = cleaned[4:].strip()

    try:
        json.loads(cleaned)
        return cleaned
    except json.JSONDecodeError:
        pass

    object_start = cleaned.find("{")
    array_start = cleaned.find("[")
    starts = [index for index in (object_start, array_start) if index != -1]
    if not starts:
        raise ValueError("返回内容中未找到 JSON 起始符")
    start = min(starts)
    end = max(cleaned.rfind("}"), cleaned.rfind("]"))
    if end == -1 or end <= start:
        raise ValueError("返回内容中未找到 JSON 结束符")
    candidate = cleaned[start:end + 1]
    json.loads(candidate)
    return candidate


def parse_json_response(raw_text: str) -> dict:
    parsed = json.loads(extract_json_text(raw_text))
    if not isinstance(parsed, dict):
        raise ValueError("模型返回不是 JSON 对象")
    return parsed


def is_empty_list_like(value) -> bool:
    if value is None or value == 0:
        return True
    if isinstance(value, str):
        return value.strip() in {"", "0", "[]", "[0]", "null", "None"}
    if isinstance(value, list):
        if not value:
            return True
        return all(is_empty_list_like(item) for item in value)
    return False


def normalize_text_list(value) -> list[str]:
    if is_empty_list_like(value):
        return []
    if isinstance(value, list):
        items: list[str] = []
        for item in value:
            if is_empty_list_like(item):
                continue
            if isinstance(item, (dict, list)):
                items.append(json.dumps(item, ensure_ascii=False))
            else:
                text = str(item).strip()
                if text:
                    items.append(text)
        return items
    text = str(value).strip()
    return [text] if text else []


def normalize_object_list(value) -> list[dict]:
    if is_empty_list_like(value):
        return []
    if isinstance(value, dict):
        return [value] if value else []
    if not isinstance(value, list):
        return []

    objects: list[dict] = []
    for item in value:
        if is_empty_list_like(item):
            continue
        if isinstance(item, dict) and item:
            objects.append(item)
    return objects


def format_string_list(value) -> str:
    items = normalize_text_list(value)
    return "[]" if not items else "; ".join(items)


def format_object_list(value) -> str:
    items = normalize_object_list(value)
    return "[]" if not items else json.dumps(items, ensure_ascii=False)


def format_scalar_text(value) -> str:
    if value is None:
        return ""
    return str(value).strip()


def extract_names(items, key: str) -> str:
    normalized_items = normalize_object_list(items)
    names: list[str] = []
    for item in normalized_items:
        name = item.get(key)
        if name is None:
            continue
        text = str(name).strip()
        if text:
            names.append(text)
    if not names:
        return "[]"
    return "; ".join(dict.fromkeys(names))


def flatten_image_result(parsed: dict) -> dict:
    analysis_list = parsed.get("image_analysis_per_image")
    if isinstance(analysis_list, list) and analysis_list:
        image_result = analysis_list[0] if isinstance(analysis_list[0], dict) else {}
    else:
        image_result = {}

    return {
        "image_summary": format_scalar_text(image_result.get("image_summary")),
        "visible_species_in_image": format_string_list(image_result.get("visible_species_in_image")),
        "landscape_elements": format_string_list(image_result.get("landscape_elements")),
        "human_activities_in_image": format_string_list(image_result.get("human_activities_in_image")),
        "plants_detected_names": extract_names(image_result.get("plants_detected"), "scientific_name"),
        "animals_detected_names": extract_names(image_result.get("animals_detected"), "scientific_name"),
        "human_activities_detected_labels": extract_names(image_result.get("human_activities_detected"), "activity"),
        "plants_detected_raw": format_object_list(image_result.get("plants_detected")),
        "animals_detected_raw": format_object_list(image_result.get("animals_detected")),
        "human_activities_detected_raw": format_object_list(image_result.get("human_activities_detected")),
    }


instruction = load_instruction(PROMPT_PATH)
image_paths = sorted(
    path for path in IMAGE_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
)

if MAX_IMAGES is not None:
    image_paths = image_paths[:MAX_IMAGES]

print(f"待处理图片数：{len(image_paths)}")

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
records: list[dict] = []

for image_index, image_path in enumerate(image_paths, start=1):
    raw_response = ""
    base_record = {
        "image_index": image_index,
        "image_filename": image_path.name,
    }
    if KEEP_IMAGE_PATH_COLUMN:
        base_record["image_path"] = str(image_path)

    try:
        image_data_url = image_file_to_data_url(str(image_path))
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": instruction},
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": (

                                "All output text must be in English only.\n"
                                "Never output Chinese text.\n"
                                "For any list or detection module with no result, return [] exactly.\n"
                                "Output only the required JSON."
                            ),
                        },
                        {
                            "type": "image_url",
                            "image_url": {"url": image_data_url},
                        },
                    ],
                },
            ],
            max_tokens=MAX_TOKENS,
            temperature=TEMPERATURE,
            top_p=0.8,
            presence_penalty=1.5,
            extra_body={
                "top_k": 20,
                "chat_template_kwargs": {"enable_thinking": False},},
        )
        raw_response = message_content_to_text(response.choices[0].message.content)
        parsed = parse_json_response(raw_response)
        records.append(
            {
                **base_record,
                **flatten_image_result(parsed),
                "raw_response": raw_response,
                "parse_ok": True,
                "error": "",
            }
        )
    except Exception as exc:
        records.append(
            {
                **base_record,
                **flatten_image_result({}),
                "raw_response": raw_response,
                "parse_ok": False,
                "error": str(exc),
            }
        )

    if REQUEST_INTERVAL_SECONDS:
        time.sleep(REQUEST_INTERVAL_SECONDS)

    if image_index % 10 == 0 or image_index == len(image_paths):
        print(f"已完成 {image_index}/{len(image_paths)} 张图片")

results_df = pd.DataFrame(records)
ordered_columns = [
    "image_index",
    "image_filename",
    *( ["image_path"] if KEEP_IMAGE_PATH_COLUMN else [] ),
    "image_summary",
    "visible_species_in_image",
    "landscape_elements",
    "human_activities_in_image",
    "plants_detected_names",
    "animals_detected_names",
    "human_activities_detected_labels",
    "plants_detected_raw",
    "animals_detected_raw",
    "human_activities_detected_raw",
    "raw_response",
    "parse_ok",
    "error",
]
results_df = results_df.reindex(columns=ordered_columns)
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"结果已保存到：{OUTPUT_CSV}")
results_df.head()

待处理图片数：300
已完成 10/300 张图片
已完成 20/300 张图片
已完成 30/300 张图片
已完成 40/300 张图片
已完成 50/300 张图片
已完成 60/300 张图片
已完成 70/300 张图片
已完成 80/300 张图片
已完成 90/300 张图片
已完成 100/300 张图片
已完成 110/300 张图片
已完成 120/300 张图片
已完成 130/300 张图片
已完成 140/300 张图片
已完成 150/300 张图片
已完成 160/300 张图片
已完成 170/300 张图片
已完成 180/300 张图片
已完成 190/300 张图片
已完成 200/300 张图片
已完成 210/300 张图片
已完成 220/300 张图片
已完成 230/300 张图片
已完成 240/300 张图片
已完成 250/300 张图片
已完成 260/300 张图片
已完成 270/300 张图片
已完成 280/300 张图片
已完成 290/300 张图片
已完成 300/300 张图片
结果已保存到：/home/dream/Study/26p1/DSproject/ProjectNew/examples/output/image_instruction_results3-5.csv


,image_index,image_filename,image_summary,visible_species_in_image,landscape_elements,human_activities_in_image,plants_detected_names,animals_detected_names,human_activities_detected_labels,plants_detected_raw,animals_detected_raw,human_activities_detected_raw,raw_response,parse_ok,error
0,1,1天津市蓟州区盘山_110296_1.jpg,stone wall with calligraphy,[],sky; stone wall; barrier,[],unknown,[],[],"[{""scientific_name"": ""unknown"", ""count_estimat...",[],[],"{\n ""image_analysis_per_image"": [\n {\n ...",True,
1,2,1天津市蓟州区盘山_110296_2.jpg,temple courtyard,[],roof; trees; stairs; stone wall,visiting; walking,unknown,[],visiting,"[{""scientific_name"": ""unknown"", ""count_estimat...",[],"[{""activity"": ""visiting"", ""count_estimate"": ""3...","{\n ""image_analysis_per_image"": [\n {\n ...",True,
2,3,1天津市蓟州区盘山_110296_3.jpg,Mountain view with pagoda,Pinus species,mountain; pagoda; sky; trees; rock,[],Pinus,[],[],"[{""scientific_name"": ""Pinus"", ""count_estimate""...",[],[],"{\n ""image_analysis_per_image"": [\n {\n ...",True,
3,4,1天津市蓟州区盘山_110296_4.jpg,Cable car on mountain,[],mountains; trees; sky; cables,riding cable car,[],[],riding cable car,[],[],"[{""activity"": ""riding cable car"", ""count_estim...","{\n ""image_analysis_per_image"": [\n {\n ...",True,
4,5,1天津市蓟州区盘山_110296_5.jpg,stone wall with calligraphy,[],stone wall; sky; flags,[],[],[],[],[],[],[],"{\n ""image_analysis_per_image"": [\n {\n ...",True,
